In [5]:
#!/usr/bin/env python3
"""
Plot segment summary statistics alongshore for Texas shoreline segments.

Reads:
    F:/crs/proj/2026_shoreline_analysis/segment_analysis_by_inlet/segment_summary_report.csv
    F:/crs/proj/2026_shoreline_analysis/place_names_short.csv
    F:/crs/proj/2026_shoreline_analysis/processed/tx_shoreline_stats.parquet

Makes stacked alongshore plots with:
- x = alongshore distance from the SW end of the coast
- each segment drawn across its full alongshore extent
- mean shown as a level line across the segment
- shaded band = mean +/- std across the segment
- min and max shown as horizontal dashed lines across the segment
- segment boundaries shown explicitly
- segment labels assigned from place_names_short.csv entries where Inlet == "N",
  located by nearest segment midpoint in lat/lon space

This version uses:
- one single color per subplot for mean, std shading, min, and max
- no alternating segment shading
- place_names_short.csv instead of place_names.csv
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# FILES
# ============================================================

SUMMARY_FILE = Path(
    r"F:/crs/proj/2026_shoreline_analysis/segment_analysis_by_inlet/segment_summary_report.csv"
)

PLACE_NAMES_FILE = Path(
    r"F:/crs/proj/2026_shoreline_analysis/place_names_short.csv"
)

STATS_FILE = Path(
    r"F:/crs/proj/2026_shoreline_analysis/processed/tx_shoreline_stats.parquet"
)

OUTDIR = SUMMARY_FILE.parent
OUTFILE = OUTDIR / "segment_summary_alongshore_by_segment.png"


# ============================================================
# USER SETTINGS
# ============================================================

FIGSIZE = (13, 12)
BOUNDARY_ALPHA = 0.20
BAND_ALPHA = 0.25
LINEWIDTH = 2.0
DASHED_LW = 1.2
SEGMENT_LABEL_Y = 0.98  # axes fraction on top panel

# Transect geometry columns in STATS_FILE
ID_COL = "ID"
LON_COL = "lon_mid"
LAT_COL = "lat_mid"
S_KM_COL = "s_km"

# Panels to plot: (mean, std, min, max, ylabel)
PANEL_GROUPS = [
    [
        ("rate_ols_mean", "rate_ols_std", "rate_ols_min", "rate_ols_max", "OLS rate (m/yr)"),
        ("rate_ts_mean", "rate_ts_std", "rate_ts_min", "rate_ts_max", "TS rate (m/yr)"),
    ],
    [
        ("var_trend_mean", "var_trend_std", "var_trend_min", "var_trend_max", "Trend variance"),
        ("var_seasonal_mean", "var_seasonal_std", "var_seasonal_min", "var_seasonal_max", "Seasonal variance"),
        ("var_resid_mean", "var_resid_std", "var_resid_min", "var_resid_max", "Residual variance"),
        ("var_total_grid_mean", "var_total_grid_std", "var_total_grid_min", "var_total_grid_max", "Total variance"),
    ],
    [
        ("pct_trend_mean", "pct_trend_std", "pct_trend_min", "pct_trend_max", "Trend (%)"),
        ("pct_seasonal_mean", "pct_seasonal_std", "pct_seasonal_min", "pct_seasonal_max", "Seasonal (%)"),
        ("pct_resid_mean", "pct_resid_std", "pct_resid_min", "pct_resid_max", "Residual (%)"),
    ],
]


# ============================================================
# HELPERS
# ============================================================

def falsey_inlet(x) -> bool:
    if pd.isna(x):
        return False
    return str(x).strip().upper() == "N"


def haversine_m(lon1, lat1, lon2, lat2):
    """
    Great-circle distance in meters.
    Supports numpy arrays.
    """
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return 6371000.0 * c


def prep_segment_summary(summary_file: str | Path) -> pd.DataFrame:
    df = pd.read_csv(summary_file)

    for col in ["s_km_min", "s_km_max", "segment_id"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # convert to distance from SW
    s_global_max = np.nanmax(np.r_[df["s_km_min"].to_numpy(), df["s_km_max"].to_numpy()])
    df["x0_sw"] = s_global_max - df["s_km_max"]   # smaller x = SW edge of segment
    df["x1_sw"] = s_global_max - df["s_km_min"]   # larger x = NE edge of segment
    df["xmid_sw"] = 0.5 * (df["x0_sw"] + df["x1_sw"])

    # sort SW -> NE
    df = df.sort_values("xmid_sw", ascending=True).reset_index(drop=True)

    return df


def load_segment_midpoints_from_stats(
    summary_df: pd.DataFrame,
    stats_file: str | Path,
) -> pd.DataFrame:
    """
    Compute geographic midpoint (lon/lat) for each segment by selecting transects
    whose s_km falls within [s_km_min, s_km_max] from the stats file, then taking
    the median lon/lat of those transects.
    """
    stats = pd.read_parquet(stats_file)

    needed = [ID_COL, LON_COL, LAT_COL, S_KM_COL]
    missing = [c for c in needed if c not in stats.columns]
    if missing:
        raise ValueError(f"STATS_FILE is missing required columns: {missing}")

    tran = (
        stats[needed]
        .drop_duplicates(subset=[ID_COL])
        .copy()
    )

    tran[S_KM_COL] = pd.to_numeric(tran[S_KM_COL], errors="coerce")
    tran[LON_COL] = pd.to_numeric(tran[LON_COL], errors="coerce")
    tran[LAT_COL] = pd.to_numeric(tran[LAT_COL], errors="coerce")

    seg_lon = []
    seg_lat = []
    n_tran = []

    for _, row in summary_df.iterrows():
        s0 = pd.to_numeric(pd.Series([row["s_km_min"]]), errors="coerce").iloc[0]
        s1 = pd.to_numeric(pd.Series([row["s_km_max"]]), errors="coerce").iloc[0]

        if not (np.isfinite(s0) and np.isfinite(s1)):
            seg_lon.append(np.nan)
            seg_lat.append(np.nan)
            n_tran.append(0)
            continue

        lo = min(s0, s1)
        hi = max(s0, s1)

        use = tran.loc[(tran[S_KM_COL] >= lo) & (tran[S_KM_COL] <= hi)].copy()

        if len(use) == 0:
            seg_lon.append(np.nan)
            seg_lat.append(np.nan)
            n_tran.append(0)
            continue

        seg_lon.append(float(np.nanmedian(use[LON_COL].to_numpy())))
        seg_lat.append(float(np.nanmedian(use[LAT_COL].to_numpy())))
        n_tran.append(int(len(use)))

    out = summary_df.copy()
    out["segment_lon_mid"] = seg_lon
    out["segment_lat_mid"] = seg_lat
    out["segment_n_transects_from_stats"] = n_tran

    return out


def load_noninlet_places(place_file: str | Path) -> pd.DataFrame:
    """
    Return place names where Inlet == 'N' with lat/lon.
    """
    place = pd.read_csv(place_file)

    needed = ["Place_name", "Latitude", "Longitude", "Inlet"]
    missing = [c for c in needed if c not in place.columns]
    if missing:
        raise ValueError(f"place_names_short.csv is missing required columns: {missing}")

    mask = place["Inlet"].map(falsey_inlet)
    out = place.loc[mask, ["Place_name", "Latitude", "Longitude"]].copy()

    out["Latitude"] = pd.to_numeric(out["Latitude"], errors="coerce")
    out["Longitude"] = pd.to_numeric(out["Longitude"], errors="coerce")

    out = out.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)
    return out


def assign_segment_labels_by_location(
    summary_df: pd.DataFrame,
    place_file: str | Path,
) -> pd.DataFrame:
    """
    Assign one non-inlet place name to each segment by nearest geographic midpoint.

    This uses a one-to-one greedy assignment:
    - compute distance from each segment midpoint to each non-inlet place
    - repeatedly assign the closest unmatched pair

    Unmatched segments fall back to 'Segment <id>'.
    """
    df = summary_df.copy()
    places = load_noninlet_places(place_file)

    labels = [None] * len(df)

    good_seg = df["segment_lon_mid"].notna() & df["segment_lat_mid"].notna()
    seg_idx = np.where(good_seg.to_numpy())[0]
    plc_idx = np.arange(len(places))

    if len(seg_idx) == 0 or len(plc_idx) == 0:
        df["segment_label"] = [
            f"Segment {int(sid)}" if pd.notna(sid) else f"Segment {i+1}"
            for i, sid in enumerate(df.get("segment_id", pd.Series(np.arange(1, len(df) + 1))))
        ]
        return df

    D = np.full((len(seg_idx), len(plc_idx)), np.nan)

    for i, iseg in enumerate(seg_idx):
        slon = df.loc[iseg, "segment_lon_mid"]
        slat = df.loc[iseg, "segment_lat_mid"]
        D[i, :] = haversine_m(
            slon,
            slat,
            places["Longitude"].to_numpy(),
            places["Latitude"].to_numpy(),
        )

    used_seg = set()
    used_plc = set()

    while True:
        best = None
        best_d = np.inf

        for i in range(D.shape[0]):
            if i in used_seg:
                continue
            for j in range(D.shape[1]):
                if j in used_plc:
                    continue
                dij = D[i, j]
                if np.isfinite(dij) and dij < best_d:
                    best_d = dij
                    best = (i, j)

        if best is None:
            break

        i, j = best
        used_seg.add(i)
        used_plc.add(j)

        iseg = seg_idx[i]
        labels[iseg] = places.loc[j, "Place_name"]

    for i in range(len(df)):
        if labels[i] is None:
            sid = df.loc[i, "segment_id"] if "segment_id" in df.columns else i + 1
            labels[i] = f"Segment {int(sid)}"

    df["segment_label"] = labels
    return df


def draw_segment_boundaries(ax, df: pd.DataFrame, label_segments: bool = False):
    """
    Draw segment boundary lines and optional segment labels.
    """
    for _, row in df.iterrows():
        x0 = row["x0_sw"]
        x1 = row["x1_sw"]

        if np.isfinite(x0):
            ax.axvline(x0, lw=0.8, alpha=BOUNDARY_ALPHA, color="0.3")
        if np.isfinite(x1):
            ax.axvline(x1, lw=0.8, alpha=BOUNDARY_ALPHA, color="0.3")

        if label_segments and np.isfinite(x0) and np.isfinite(x1):
            label = row.get("segment_label", f"Segment {int(row['segment_id'])}")
            ax.text(
                0.5 * (x0 + x1),
                SEGMENT_LABEL_Y,
                str(label),
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="top",
                fontsize=8,
                rotation=0,
                alpha=0.9,
            )


def plot_segment_level_lines(
    ax,
    df: pd.DataFrame,
    mean_col: str,
    std_col: str,
    min_col: str,
    max_col: str,
    ylabel: str,
    show_legend: bool = False,
):
    """
    Plot each segment as an independent horizontal level line over its full x extent.

    Uses one single color per subplot for:
    - mean solid line
    - std shading
    - min dashed line
    - max dashed line
    """
    draw_segment_boundaries(ax, df, label_segments=False)

    # Use one matplotlib default color per subplot
    panel_color = ax._get_lines.get_next_color()

    added_mean = False
    added_band = False
    added_min = False
    added_max = False

    for _, row in df.iterrows():
        x0 = pd.to_numeric(pd.Series([row["x0_sw"]]), errors="coerce").iloc[0]
        x1 = pd.to_numeric(pd.Series([row["x1_sw"]]), errors="coerce").iloc[0]

        ym = pd.to_numeric(pd.Series([row.get(mean_col, np.nan)]), errors="coerce").iloc[0]
        ys = pd.to_numeric(pd.Series([row.get(std_col, np.nan)]), errors="coerce").iloc[0]
        yn = pd.to_numeric(pd.Series([row.get(min_col, np.nan)]), errors="coerce").iloc[0]
        yx = pd.to_numeric(pd.Series([row.get(max_col, np.nan)]), errors="coerce").iloc[0]

        if not (np.isfinite(x0) and np.isfinite(x1)):
            continue

        if np.isfinite(ym) and np.isfinite(ys):
            ax.fill_between(
                [x0, x1],
                [ym - ys, ym - ys],
                [ym + ys, ym + ys],
                alpha=BAND_ALPHA,
                color=panel_color,
                label="mean ± std" if (show_legend and not added_band) else None,
            )
            added_band = True

        if np.isfinite(ym):
            ax.plot(
                [x0, x1],
                [ym, ym],
                "-",
                lw=LINEWIDTH,
                color=panel_color,
                label="mean" if (show_legend and not added_mean) else None,
            )
            added_mean = True

        if np.isfinite(yn):
            ax.plot(
                [x0, x1],
                [yn, yn],
                "--",
                lw=DASHED_LW,
                color=panel_color,
                label="min" if (show_legend and not added_min) else None,
            )
            added_min = True

        if np.isfinite(yx):
            ax.plot(
                [x0, x1],
                [yx, yx],
                "--",
                lw=DASHED_LW,
                color=panel_color,
                label="max" if (show_legend and not added_max) else None,
            )
            added_max = True

    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)


def make_stacked_plots(df: pd.DataFrame, outfile: str | Path):
    n_panels = sum(len(group) for group in PANEL_GROUPS)

    fig, axs = plt.subplots(
        n_panels,
        1,
        figsize=FIGSIZE,
        sharex=True,
        constrained_layout=True,
    )

    if n_panels == 1:
        axs = [axs]

    iax = 0
    for group in PANEL_GROUPS:
        for mean_col, std_col, min_col, max_col, ylabel in group:
            ax = axs[iax]

            plot_segment_level_lines(
                ax,
                df,
                mean_col=mean_col,
                std_col=std_col,
                min_col=min_col,
                max_col=max_col,
                ylabel=ylabel,
                show_legend=(iax == 0),
            )

            if iax == 0:
                ax.set_title("Segment summary statistics alongshore")

            iax += 1

    draw_segment_boundaries(axs[0], df, label_segments=True)

    axs[-1].set_xlabel("Alongshore distance from SW (km)")

    xmin = np.nanmin(df["x0_sw"].to_numpy())
    xmax = np.nanmax(df["x1_sw"].to_numpy())
    axs[-1].set_xlim(xmin, xmax)

    handles, labels = axs[0].get_legend_handles_labels()
    if handles:
        axs[0].legend(handles, labels, loc="best")

    fig.savefig(outfile, dpi=200)
    plt.close(fig)


def make_group_plot(df: pd.DataFrame, group, title: str, outfile: str | Path):
    fig, axs = plt.subplots(
        len(group),
        1,
        figsize=(13, 2.8 * len(group)),
        sharex=True,
        constrained_layout=True,
    )

    if len(group) == 1:
        axs = [axs]

    for iax, (ax, (mean_col, std_col, min_col, max_col, ylabel)) in enumerate(zip(axs, group)):
        plot_segment_level_lines(
            ax,
            df,
            mean_col=mean_col,
            std_col=std_col,
            min_col=min_col,
            max_col=max_col,
            ylabel=ylabel,
            show_legend=(iax == 0),
        )

    draw_segment_boundaries(axs[0], df, label_segments=True)

    axs[0].set_title(title)
    axs[-1].set_xlabel("Alongshore distance from SW (km)")

    xmin = np.nanmin(df["x0_sw"].to_numpy())
    xmax = np.nanmax(df["x1_sw"].to_numpy())
    axs[-1].set_xlim(xmin, xmax)

    handles, labels = axs[0].get_legend_handles_labels()
    if handles:
        axs[0].legend(handles, labels, loc="best")

    fig.savefig(outfile, dpi=200)
    plt.close(fig)

def make_ts_rate_plot(df: pd.DataFrame, outfile: str | Path):
    """
    Make a single-panel alongshore plot showing only TS rate.

    Expects df to already contain:
        x0_sw, x1_sw, xmid_sw, segment_label
        rate_ts_mean, rate_ts_std, rate_ts_min, rate_ts_max
    """
    fig, ax = plt.subplots(
        1, 1,
        figsize=(13, 4.5),
        constrained_layout=True,
    )

    # boundary lines
    for _, row in df.iterrows():
        x0 = pd.to_numeric(pd.Series([row["x0_sw"]]), errors="coerce").iloc[0]
        x1 = pd.to_numeric(pd.Series([row["x1_sw"]]), errors="coerce").iloc[0]

        if np.isfinite(x0):
            ax.axvline(x0, lw=0.8, alpha=BOUNDARY_ALPHA, color="0.3")
        if np.isfinite(x1):
            ax.axvline(x1, lw=0.8, alpha=BOUNDARY_ALPHA, color="0.3")

    # one single color for this subplot
    panel_color = ax._get_lines.get_next_color()

    added_mean = False
    added_band = False
    added_min = False
    added_max = False

    for _, row in df.iterrows():
        x0 = pd.to_numeric(pd.Series([row["x0_sw"]]), errors="coerce").iloc[0]
        x1 = pd.to_numeric(pd.Series([row["x1_sw"]]), errors="coerce").iloc[0]

        ym = pd.to_numeric(pd.Series([row.get("rate_ts_mean", np.nan)]), errors="coerce").iloc[0]
        ys = pd.to_numeric(pd.Series([row.get("rate_ts_std", np.nan)]), errors="coerce").iloc[0]
        yn = pd.to_numeric(pd.Series([row.get("rate_ts_min", np.nan)]), errors="coerce").iloc[0]
        yx = pd.to_numeric(pd.Series([row.get("rate_ts_max", np.nan)]), errors="coerce").iloc[0]

        if not (np.isfinite(x0) and np.isfinite(x1)):
            continue

        # std shading
        if np.isfinite(ym) and np.isfinite(ys):
            ax.fill_between(
                [x0, x1],
                [ym - ys, ym - ys],
                [ym + ys, ym + ys],
                alpha=BAND_ALPHA,
                color=panel_color,
                label="mean ± std" if not added_band else None,
            )
            added_band = True

        # mean line
        if np.isfinite(ym):
            ax.plot(
                [x0, x1],
                [ym, ym],
                "-",
                lw=LINEWIDTH,
                color=panel_color,
                label="TS mean" if not added_mean else None,
            )
            added_mean = True

        # min dashed line
        if np.isfinite(yn):
            ax.plot(
                [x0, x1],
                [yn, yn],
                "--",
                lw=DASHED_LW,
                color=panel_color,
                label="min" if not added_min else None,
            )
            added_min = True

        # max dashed line
        if np.isfinite(yx):
            ax.plot(
                [x0, x1],
                [yx, yx],
                "--",
                lw=DASHED_LW,
                color=panel_color,
                label="max" if not added_max else None,
            )
            added_max = True

    # vertical island labels along top
    for _, row in df.iterrows():
        x0 = pd.to_numeric(pd.Series([row["x0_sw"]]), errors="coerce").iloc[0]
        x1 = pd.to_numeric(pd.Series([row["x1_sw"]]), errors="coerce").iloc[0]
        if np.isfinite(x0) and np.isfinite(x1):
            ax.text(
                0.5 * (x0 + x1),
                0.98,
                str(row.get("segment_label", "")),
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="top",
                rotation=90,
                fontsize=8,
                alpha=0.9,
            )

    ax.set_title("Segment TS change rates alongshore")
    ax.set_xlabel("Alongshore distance from SW (km)")
    ax.set_ylabel("TS rate (m/yr)")
    ax.grid(True, alpha=0.3)

    xmin = np.nanmin(df["x0_sw"].to_numpy())
    xmax = np.nanmax(df["x1_sw"].to_numpy())
    ax.set_xlim(xmin, xmax)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc="best")

    fig.savefig(outfile, dpi=200)
    plt.close(fig)


# ============================================================
# MAIN
# ============================================================

def main():
    df = prep_segment_summary(SUMMARY_FILE)
    df = load_segment_midpoints_from_stats(df, STATS_FILE)
    df = assign_segment_labels_by_location(df, PLACE_NAMES_FILE)

    make_stacked_plots(df, OUTFILE)

    make_group_plot(
        df,
        PANEL_GROUPS[0],
        title="Segment change rates alongshore",
        outfile=OUTDIR / "segment_summary_rates_alongshore.png",
    )

    make_ts_rate_plot(
        df,
        OUTDIR / "TS_segment_summary_rates_alongshore.png",
    )
    
    make_group_plot(
        df,
        PANEL_GROUPS[1],
        title="Segment variance statistics alongshore",
        outfile=OUTDIR / "segment_summary_variance_alongshore.png",
    )

    make_group_plot(
        df,
        PANEL_GROUPS[2],
        title="Segment variance percentages alongshore",
        outfile=OUTDIR / "segment_summary_percent_alongshore.png",
    )

    df.to_csv(OUTDIR / "segment_summary_report_labeled.csv", index=False)

    print(f"Wrote: {OUTFILE}")
    print(f"Wrote: {OUTDIR / 'segment_summary_rates_alongshore.png'}")
    print(f"Wrote: {OUTDIR / 'segment_summary_variance_alongshore.png'}")
    print(f"Wrote: {OUTDIR / 'segment_summary_percent_alongshore.png'}")
    print(f"Wrote: {OUTDIR / 'segment_summary_report_labeled.csv'}")


if __name__ == "__main__":
    main()

Wrote: F:\crs\proj\2026_shoreline_analysis\segment_analysis_by_inlet\segment_summary_alongshore_by_segment.png
Wrote: F:\crs\proj\2026_shoreline_analysis\segment_analysis_by_inlet\segment_summary_rates_alongshore.png
Wrote: F:\crs\proj\2026_shoreline_analysis\segment_analysis_by_inlet\segment_summary_variance_alongshore.png
Wrote: F:\crs\proj\2026_shoreline_analysis\segment_analysis_by_inlet\segment_summary_percent_alongshore.png
Wrote: F:\crs\proj\2026_shoreline_analysis\segment_analysis_by_inlet\segment_summary_report_labeled.csv
